# Prueba corta: elegir la red preentrenada

Etapa 1, paso 5. Compara **MobileNetV3Large** y **EfficientNetB0** con la base congelada y pocas épocas, sobre el mismo split. Se evalúa en **validación**; test se reserva para el modelo final.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

Necesitas en tu Google Drive el zip del dataset en `MyDrive/Datalab/2025-05-10_archive.zip` (cambia la ruta abajo si lo tienes en otro lado).

In [ ]:
# 1. Montar Drive y traer el repo
from google.colab import drive
drive.mount('/content/drive')

RUTA_ZIP = '/content/drive/MyDrive/Datalab/2025-05-10_archive.zip'  # <- ajusta si hace falta

# Si el repo es privado, usa un token: https://<TOKEN>@github.com/itam-datalab/clasificacion_basura.git
!git clone https://github.com/itam-datalab/clasificacion_basura.git
%cd clasificacion_basura
!pip install -q -r requirements.txt

In [ ]:
# 2. Preparar datos (mismo split que el que está en datos/, porque la semilla es fija)
!mkdir -p Datasets && cp "$RUTA_ZIP" Datasets/2025-05-10_archive.zip
!cp datos/splits.csv /tmp/splits_repo.csv 2>/dev/null || echo 'datos/ aún no está en el repo'
!python src/preparar_datos.py
# Si datos/ ya estaba en el repo, esto debe decir que son iguales:
!cmp -s datos/splits.csv /tmp/splits_repo.csv && echo 'Split idéntico al del repo' || echo 'OJO: el split cambió o no había uno en el repo'

In [ ]:
# 3. Prueba corta (~30-60 min con GPU T4)
!python src/prueba_redes.py --bases MobileNetV3Large EfficientNetB0 --epocas 8

In [ ]:
# 4. Resultados
import pandas as pd
from IPython.display import Image, display

res = pd.read_csv('resultados/prueba_redes/resumen.csv')
display(res)
display(pd.read_csv('resultados/prueba_redes/f1_por_clase.csv'))
for red in res['red']:
    display(Image(f'resultados/prueba_redes/curvas_{red}.png'))

## Cómo elegir

1. **F1 macro en validación**: el que más alto tenga.
2. Si la diferencia es menor a ~0.02, desempata con **F1 de biological** (la clase más débil) y con **tiempo en TFLite** (más rápido es mejor para la placa).
3. El tiempo `tflite_ms_cpu` se mide en el CPU de Colab: sirve para comparar entre redes, no es el tiempo real en la placa de Makers.

Anota la decisión en el README y descarga la carpeta de resultados (celda de abajo) para subir `resumen.csv`, `f1_por_clase.csv` y las curvas al repo. Los `.keras` y `.tflite` de esta prueba no hace falta subirlos.

In [ ]:
# 5. Descargar resultados (sin los modelos)
!cd resultados && zip -r /content/prueba_redes.zip prueba_redes -x '*.keras' '*.tflite'
from google.colab import files
files.download('/content/prueba_redes.zip')